# Thalamus Framing Paper — Analysis Notebook

**Purpose**: Single source of truth for all paper figures. Every figure can be regenerated from scratch by running the cells top-to-bottom.

**Data provenance**:
- `all_sentences_final.csv` — 55,848 full-text thalamus sentences with BERTopic assignments and framing labels
- `framing_combined.csv` — 2,536 framing sentences (full-text + abstracts)
- `framing_clean.csv` — 1,609 full-text framing sentences
- `framing_clean_abstracts.csv` — 929 abstract framing sentences
- `conclusion_themes_v2.csv` — conclusion section sentences with 10 binary theme columns

**Sections**:
0. Setup (imports, paths, shared utilities)
1. Fig 2 — Research Topics (BERTopic)
2. Fig 3 — Framing Over Time
3. Fig 4 — Framing × Nuclei & Techniques
4. Fig 5 — Nucleus Similarity Clustering
5. Fig 6 — Framing × Topics (chi-squared residuals)
6. Fig 7 — Techniques Over Time
7. Fig 8 — Task Extraction
8. Fig 9 — Confound Analysis (Logistic Regression)

---

## Section 0: Setup

Run this cell first every session. Sets paths, imports, shared colour palette and style.

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import sys
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats as scipy_stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS
from collections import OrderedDict
import re
import warnings
warnings.filterwarnings('ignore')

# ── Project paths ─────────────────────────────────────────────────────────────
ROOT = Path(r'C:\Users\brouw\Documents\PhD\Projects\LiteratureQueeryThalamusTime')
PROC = ROOT / 'Data' / 'processed'
PUB  = ROOT / 'Graphs' / 'pub'
CODE = ROOT / 'Code'
PUB.mkdir(parents=True, exist_ok=True)

# Add Code directory to Python path so we can import local modules
sys.path.insert(0, str(CODE))

# ── Load shared modules ───────────────────────────────────────────────────────
from nucleus_patterns import NUCLEI, tag_nuclei
from technique_patterns import TECHNIQUES, tag_techniques
from stats import mann_kendall, wilson_ci

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.size': 10,
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Wong colour-blind-safe palette (9 colours + black)
WONG_COLORS = [
    '#E69F00',  # orange
    '#56B4E9',  # sky blue
    '#009E73',  # bluish green
    '#F0E442',  # yellow
    '#0072B2',  # blue
    '#D55E00',  # vermilion
    '#CC79A7',  # reddish purple
    '#999999',  # grey
    '#000000',  # black
]

# ── Convenience: save + show ──────────────────────────────────────────────────
def savefig(name, dpi=150):
    """Save current figure to PUB directory and display it."""
    path = PUB / name
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'Saved: {path}')
    plt.show()

print('Setup complete.')
print('ROOT:', ROOT)
print('Nuclei loaded:', len(NUCLEI))
print('Techniques loaded:', len(TECHNIQUES))

---
## Section 1: Figure 2 — Research Topics (BERTopic)

**Data**: `all_sentences_final.csv` — 55,848 full-text sentences with BERTopic v2 assignments.

**Method**:
1. Filter out topic `-1` (unassigned sentences BERTopic cannot classify)
2. Count sentences per topic; keep top-N by total count
3. Bin years into 5-year windows; compute proportion of each topic within each bin
4. Compute Mann-Kendall trend statistics per topic
5. Plot: (a) bar chart of top topics, (b) heatmap of proportions over time, (c) trend lines for selected topics

In [ ]:
# PARAMETERS — Fig 2 Topics
TOP_N_TOPICS = 20                     # number of top topics to show in bar chart
YEAR_BINS = list(range(1975, 2030, 5)) # 5-year bin edges
MIN_SENTENCES_PER_TOPIC = 50          # drop tiny topics from trend analysis
TOPICS_TO_SHOW_TREND = [              # topic labels (partial match ok) for trend panel
    'seizures epilepsy',
    'pain nociceptive',
    'sleep wakefulness',
    'visual processing',
    'fear conditioning',
]

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df_all = pd.read_csv(PROC / 'all_sentences_final.csv', low_memory=False)
print(f'Loaded all_sentences_final.csv: {df_all.shape}')
print('Columns:', list(df_all.columns))
print('Year range:', df_all['year'].min(), '–', df_all['year'].max())

In [ ]:
# ── Filter and prepare topics ─────────────────────────────────────────────────
# Remove unassigned topic (-1)
df_topics = df_all[df_all['bertopic_id_v2'] != -1].copy()

# Use topic_clean_v2 as the display label; strip numeric prefix for readability
df_topics['topic_label'] = df_topics['topic_clean_v2'].astype(str)

# Count sentences per topic
topic_counts = (
    df_topics.groupby(['bertopic_id_v2', 'topic_label'])
    .size()
    .reset_index(name='n_sentences')
    .sort_values('n_sentences', ascending=False)
)
print(f'Total assigned topics: {len(topic_counts)}')
print(f'Topics with >= {MIN_SENTENCES_PER_TOPIC} sentences: '
      f'{(topic_counts.n_sentences >= MIN_SENTENCES_PER_TOPIC).sum()}')

In [ ]:
# ── Compute 5-year bin proportions per topic ──────────────────────────────────
# Assign each sentence to a 5-year bin
df_topics['year_bin'] = pd.cut(
    df_topics['year'],
    bins=YEAR_BINS,
    right=False,
    labels=[f'{y}–{y+4}' for y in YEAR_BINS[:-1]]
)

# Total sentences per year bin (denominator)
bin_totals = df_topics.groupby('year_bin').size().rename('bin_total')

# Sentences per topic per year bin
topic_bin = (
    df_topics.groupby(['year_bin', 'bertopic_id_v2', 'topic_label'])
    .size()
    .reset_index(name='n')
)
topic_bin = topic_bin.join(bin_totals, on='year_bin')
topic_bin['proportion'] = topic_bin['n'] / topic_bin['bin_total']

print('Topic × year-bin proportions computed.')
print(topic_bin.head())

In [ ]:
# ── Mann-Kendall trend per topic ──────────────────────────────────────────────
# Only analyse topics with enough sentences overall
big_topics = topic_counts[topic_counts['n_sentences'] >= MIN_SENTENCES_PER_TOPIC]

mk_results = []
for _, row in big_topics.iterrows():
    tid = row['bertopic_id_v2']
    subset = topic_bin[topic_bin['bertopic_id_v2'] == tid].sort_values('year_bin')
    # Use bin mid-year as x-axis for Mann-Kendall
    bin_mid = [YEAR_BINS[i] + 2 for i in range(len(YEAR_BINS) - 1)]
    prop_vals = subset.set_index('year_bin')['proportion'].reindex(
        [f'{y}–{y+4}' for y in YEAR_BINS[:-1]]
    ).values
    tau, p, sig, label = mann_kendall(bin_mid, prop_vals)
    mk_results.append({
        'bertopic_id_v2': tid,
        'topic_label': row['topic_label'],
        'n_sentences': row['n_sentences'],
        'tau': tau, 'p': p, 'sig': sig, 'trend': label
    })

df_mk_topics = pd.DataFrame(mk_results).sort_values('tau', ascending=False)
print('Emerging topics (tau > 0, p < 0.05):')
print(df_mk_topics[df_mk_topics['trend'] == 'emerging'][['topic_label', 'tau', 'p', 'sig']].head(10))
print('\nDeclining topics:')
print(df_mk_topics[df_mk_topics['trend'] == 'declining'][['topic_label', 'tau', 'p', 'sig']].head(10))

In [ ]:
# ── Figure 2: (a) Top-N bar chart + (b) heatmap of temporal proportions ───────
top_topics = topic_counts.head(TOP_N_TOPICS)
top_ids = top_topics['bertopic_id_v2'].tolist()

# Build proportion matrix for heatmap: rows = topics, cols = year bins
bin_labels = [f'{y}–{y+4}' for y in YEAR_BINS[:-1]]
heat_data = (
    topic_bin[topic_bin['bertopic_id_v2'].isin(top_ids)]
    .pivot_table(index='topic_label', columns='year_bin', values='proportion', fill_value=0)
    .reindex(columns=bin_labels, fill_value=0)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [1, 2]})

# Panel A: Horizontal bar chart (top-N topics by count)
ax = axes[0]
y_pos = range(len(top_topics))
ax.barh(y_pos, top_topics['n_sentences'], color=WONG_COLORS[0], edgecolor='white', linewidth=0.5)
# Add trend arrow annotations from MK results
mk_dict = df_mk_topics.set_index('bertopic_id_v2')['trend'].to_dict()
trend_symbols = {'emerging': '▲', 'declining': '▼', 'stable': '–', 'insufficient data': ''}
trend_colors  = {'emerging': '#009E73', 'declining': '#D55E00', 'stable': '#999999', 'insufficient data': '#999999'}
for i, (_, row) in enumerate(top_topics.iterrows()):
    trend = mk_dict.get(row['bertopic_id_v2'], 'insufficient data')
    ax.text(row['n_sentences'] + 20, i, trend_symbols.get(trend, ''),
            va='center', color=trend_colors.get(trend, '#999999'), fontsize=9)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(
    [t[:40] for t in top_topics['topic_label']],
    fontsize=8
)
ax.invert_yaxis()
ax.set_xlabel('Number of sentences')
ax.set_title(f'Top {TOP_N_TOPICS} BERTopic topics', fontsize=11, fontweight='bold')

# Panel B: Heatmap of proportions over time
ax2 = axes[1]
sns.heatmap(
    heat_data,
    ax=ax2,
    cmap='YlOrBr',
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Proportion of year-bin sentences', 'shrink': 0.7},
    yticklabels=[t[:40] for t in heat_data.index],
)
ax2.set_xlabel('Year bin')
ax2.set_ylabel('')
ax2.set_title('Topic prevalence over time (5-year bins)', fontsize=11, fontweight='bold')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=7)
plt.setp(ax2.get_yticklabels(), fontsize=8)

plt.tight_layout()
savefig('fig_NB_fig2.png')

In [ ]:
# ── Figure 2 supplement: trend lines for selected topics ─────────────────────
# Find topic IDs matching the TOPICS_TO_SHOW_TREND labels (case-insensitive partial match)
selected_ids = []
selected_labels = []
for query in TOPICS_TO_SHOW_TREND:
    match = topic_counts[topic_counts['topic_label'].str.contains(query, case=False, na=False)]
    if len(match) > 0:
        selected_ids.append(match.iloc[0]['bertopic_id_v2'])
        selected_labels.append(match.iloc[0]['topic_label'])

# Build a year series from bin midpoints
bin_mids = [y + 2 for y in YEAR_BINS[:-1]]

fig, ax = plt.subplots(figsize=(10, 5))
for i, (tid, lbl) in enumerate(zip(selected_ids, selected_labels)):
    subset = topic_bin[topic_bin['bertopic_id_v2'] == tid].copy()
    prop_series = subset.set_index('year_bin')['proportion'].reindex(
        [f'{y}–{y+4}' for y in YEAR_BINS[:-1]]
    ).values
    color = WONG_COLORS[i % len(WONG_COLORS)]
    ax.plot(bin_mids, prop_series, 'o-', color=color, label=lbl[:35], linewidth=1.8, markersize=5)
    # Mann-Kendall annotation
    tau, p, sig, trend = mann_kendall(bin_mids, prop_series)
    if sig not in ('n.d.',):
        ax.annotate(
            f'τ={tau:.2f}{sig}',
            xy=(bin_mids[-1], prop_series[~np.isnan(prop_series)][-1]),
            fontsize=7, color=color
        )

ax.set_xlabel('Year (bin midpoint)')
ax.set_ylabel('Proportion of all sentences in bin')
ax.set_title('Temporal trends: selected BERTopic topics', fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='upper left', framealpha=0.5)
plt.tight_layout()
savefig('fig_NB_fig2b_trends.png')

---
## Section 2: Figure 3 — Framing Over Time

**Data**: `framing_combined.csv` — 2,536 framing sentences (full-text + abstracts).

**Method**:
1. Compute annual count of each `frame_type`
2. Compute annual proportion = frame count / total framing sentences that year
3. Apply Wilson 95% CIs to each proportion
4. Apply rolling average for smooth trend lines
5. Compute Mann-Kendall per frame
6. Plot stacked area chart + selected trend lines with CI ribbons

In [ ]:
# PARAMETERS — Fig 3 Framing
FOCAL_FRAMES   = ['relay', 'hub_node', 'gateway', 'modulator', 'integrator',
                  'clinical_target', 'role_player', 'network_component']
MIN_YEAR       = 1975
MAX_YEAR       = 2026
SMOOTH_WINDOW  = 3   # rolling window for trend lines

In [ ]:
# ── Load framing data ─────────────────────────────────────────────────────────
df_framing = pd.read_csv(PROC / 'framing_combined.csv', low_memory=False)
print(f'Loaded framing_combined.csv: {df_framing.shape}')
print('Columns:', list(df_framing.columns))
print('Frame types found:')
print(df_framing['frame_type'].value_counts())

In [ ]:
# ── Annual proportions with Wilson CIs ───────────────────────────────────────
df_fr_filtered = df_framing[
    (df_framing['year'] >= MIN_YEAR) & (df_framing['year'] <= MAX_YEAR)
].copy()

# Annual total sentences (denominator)
annual_totals = df_fr_filtered.groupby('year').size().rename('total')

# Annual count per frame type
annual_counts = (
    df_fr_filtered.groupby(['year', 'frame_type'])
    .size()
    .reset_index(name='count')
)
annual_counts = annual_counts.join(annual_totals, on='year')
annual_counts['proportion'] = annual_counts['count'] / annual_counts['total']

# Compute Wilson CIs row-by-row
annual_counts[['ci_low', 'ci_high']] = annual_counts.apply(
    lambda r: pd.Series(wilson_ci(r['count'], r['total'])), axis=1
)

print('Annual frame proportions computed.')
print(annual_counts.head(10))

In [ ]:
# ── Mann-Kendall trend per frame type ─────────────────────────────────────────
mk_frames = {}
for frame in df_fr_filtered['frame_type'].dropna().unique():
    sub = annual_counts[annual_counts['frame_type'] == frame].sort_values('year')
    tau, p, sig, label = mann_kendall(sub['year'].values, sub['proportion'].values)
    mk_frames[frame] = {'tau': tau, 'p': p, 'sig': sig, 'trend': label}

print('Mann-Kendall results per frame:')
for f, res in sorted(mk_frames.items(), key=lambda x: x[1]['tau'], reverse=True):
    print(f"  {f:25s}  tau={res['tau']:.3f}  p={res['p']:.4f}  {res['sig']}  ({res['trend']})")

In [ ]:
# ── Figure 3: Stacked area + trend lines ─────────────────────────────────────
# Build pivot: years × frame types (proportions)
all_frames = [f for f in FOCAL_FRAMES if f in df_fr_filtered['frame_type'].unique()]
year_range = list(range(MIN_YEAR, MAX_YEAR + 1))

pivot_prop = (
    annual_counts[annual_counts['frame_type'].isin(all_frames)]
    .pivot_table(index='year', columns='frame_type', values='proportion', fill_value=0)
    .reindex(year_range, fill_value=0)
)

# Apply rolling smooth for trend lines
pivot_smooth = pivot_prop.rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean()

frame_colors = {f: WONG_COLORS[i % len(WONG_COLORS)] for i, f in enumerate(all_frames)}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Panel A: Stacked area
ax1.stackplot(
    pivot_prop.index,
    [pivot_prop[f] for f in all_frames],
    labels=all_frames,
    colors=[frame_colors[f] for f in all_frames],
    alpha=0.8
)
ax1.set_ylabel('Proportion of framing sentences')
ax1.set_title('Framing over time — stacked area', fontsize=11, fontweight='bold')
ax1.legend(loc='upper left', fontsize=8, framealpha=0.5, ncol=2)
ax1.set_xlim(MIN_YEAR, MAX_YEAR)

# Panel B: Trend lines (smoothed proportions) with CI ribbons for focal frames
for frame in all_frames:
    if frame not in pivot_smooth.columns:
        continue
    color = frame_colors[frame]
    sub_ci = annual_counts[annual_counts['frame_type'] == frame].sort_values('year')
    # Reindex CI to full year range
    ci_lo = sub_ci.set_index('year')['ci_low'].reindex(year_range)
    ci_hi = sub_ci.set_index('year')['ci_high'].reindex(year_range)
    trend_line = pivot_smooth[frame]
    mk = mk_frames.get(frame, {})
    lbl = f"{frame} (τ={mk.get('tau', float('nan')):.2f}{mk.get('sig', '')})"
    ax2.plot(year_range, trend_line, color=color, linewidth=2, label=lbl)
    ax2.fill_between(year_range, ci_lo, ci_hi, color=color, alpha=0.15)

ax2.set_xlabel('Year')
ax2.set_ylabel('Proportion (smoothed)')
ax2.set_title('Framing trends with 95% Wilson CIs', fontsize=11, fontweight='bold')
ax2.legend(loc='upper left', fontsize=8, framealpha=0.5, ncol=2)
ax2.set_xlim(MIN_YEAR, MAX_YEAR)

# Annotate relay-hub crossover
if 'relay' in pivot_smooth.columns and 'hub_node' in pivot_smooth.columns:
    diff = pivot_smooth['hub_node'] - pivot_smooth['relay']
    crossover_years = diff[diff > 0].index
    if len(crossover_years) > 0:
        cx = crossover_years[0]
        ax2.axvline(cx, color='black', linestyle='--', linewidth=1, alpha=0.6)
        ax2.text(cx + 0.3, ax2.get_ylim()[1] * 0.95, f'Crossover\n~{cx}',
                 fontsize=8, va='top')

plt.tight_layout()
savefig('fig_NB_fig3.png')

---
## Section 3: Figure 4 — Framing × Nuclei & Techniques

**Data**: `framing_combined.csv` with `tag_nuclei()` and `tag_techniques()` applied.

**Method**:
1. Tag each framing sentence with nucleus labels (`has_MD`, `has_VP`, …)
2. Tag each sentence with technique labels (`has_fMRI`, …)
3. For each nucleus: count sentences by `frame_type`, compute column-normalized proportions
4. Plot heatmap: rows = nuclei, columns = frame types, color = % of that nucleus's sentences using each frame

In [ ]:
# PARAMETERS — Fig 4
NUCLEI_TO_SHOW = ['MD', 'VP', 'VL', 'Pulvinar', 'LGN', 'ANT', 'CM', 'PVT',
                  'TRN', 'CL', 'POm', 'Reuniens', 'Habenula', 'MGN']
MIN_NUCLEUS_SENTENCES = 10    # minimum sentences tagged to a nucleus to include it
NORMALIZE = 'column'          # 'column' = normalize per nucleus; 'row' = per frame type

In [ ]:
# ── Tag nuclei and techniques on framing sentences ────────────────────────────
df_fr4 = df_framing.copy()
df_fr4 = tag_nuclei(df_fr4, text_col='sentence')
df_fr4 = tag_techniques(df_fr4, text_col='sentence')

# Show how many sentences each nucleus matched
nucleus_match_counts = {nid: df_fr4[f'has_{nid}'].sum() for nid in NUCLEI}
print('Nucleus match counts (framing sentences):')
for nid, cnt in sorted(nucleus_match_counts.items(), key=lambda x: -x[1]):
    print(f'  {nid:15s}: {cnt}')

In [ ]:
# ── Build Framing × Nucleus cross-tabulation ──────────────────────────────────
# Filter nuclei with enough sentences
valid_nuclei = [n for n in NUCLEI_TO_SHOW
                if nucleus_match_counts.get(n, 0) >= MIN_NUCLEUS_SENTENCES]
print(f'Nuclei included (>= {MIN_NUCLEUS_SENTENCES} sentences): {valid_nuclei}')

frames_found = df_fr4['frame_type'].dropna().unique()

# Build matrix: rows = nuclei, cols = frame types
cross_tab = pd.DataFrame(index=valid_nuclei, columns=frames_found, dtype=float)
for nid in valid_nuclei:
    subset = df_fr4[df_fr4[f'has_{nid}']]
    total = len(subset)
    for frame in frames_found:
        cross_tab.loc[nid, frame] = (subset['frame_type'] == frame).sum() / total

cross_tab = cross_tab.fillna(0)
print('Cross-tabulation shape:', cross_tab.shape)
cross_tab.head()

In [ ]:
# ── Figure 4a: Framing × Nucleus heatmap ──────────────────────────────────────
# Use nucleus display labels
row_labels = [NUCLEI[n]['label'] for n in valid_nuclei]

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    cross_tab.astype(float),
    ax=ax,
    cmap='RdYlBu_r',
    annot=True,
    fmt='.2f',
    linewidths=0.4,
    linecolor='white',
    yticklabels=row_labels,
    cbar_kws={'label': 'Fraction of nucleus sentences', 'shrink': 0.8}
)
ax.set_xlabel('Frame type')
ax.set_ylabel('Thalamic nucleus')
ax.set_title('Framing × Nucleus (proportion of nucleus sentences per frame)', fontsize=11, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), fontsize=9)

plt.tight_layout()
savefig('fig_NB_fig4a_framing_nucleus.png')

In [ ]:
# ── Figure 4b: Framing × Technique heatmap ───────────────────────────────────
technique_ids = list(TECHNIQUES.keys())
tech_match_counts = {tid: df_fr4[f'has_{tid}'].sum() for tid in technique_ids}

# Build matrix: rows = techniques, cols = frame types
valid_techniques = [t for t in technique_ids if tech_match_counts.get(t, 0) >= 5]
cross_tech = pd.DataFrame(index=valid_techniques, columns=frames_found, dtype=float)
for tid in valid_techniques:
    subset = df_fr4[df_fr4[f'has_{tid}']]
    total = len(subset)
    if total == 0:
        continue
    for frame in frames_found:
        cross_tech.loc[tid, frame] = (subset['frame_type'] == frame).sum() / total

cross_tech = cross_tech.fillna(0)
tech_row_labels = [TECHNIQUES[t]['label'] for t in valid_techniques]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    cross_tech.astype(float),
    ax=ax,
    cmap='RdYlBu_r',
    annot=True,
    fmt='.2f',
    linewidths=0.4,
    linecolor='white',
    yticklabels=tech_row_labels,
    cbar_kws={'label': 'Fraction of technique sentences', 'shrink': 0.8}
)
ax.set_xlabel('Frame type')
ax.set_ylabel('Recording technique')
ax.set_title('Framing × Technique (proportion of technique sentences per frame)', fontsize=11, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), fontsize=9)

plt.tight_layout()
savefig('fig_NB_fig4b_framing_technique.png')

---
## Section 4: Figure 5 — Nucleus Similarity Clustering

**Data**: `all_sentences_final.csv` (topics) + `framing_combined.csv` (framing), with nucleus tags applied.

**Method**: Cluster 19 nuclei by three feature vectors:
- (a) **Framing profile**: proportion of each of the detected frame types
- (b) **Topic profile**: proportion of sentences in each `mega_theme`
- (c) **Technique profile**: proportion of sentences mentioning each of 11 techniques

Ward linkage + cosine distance → dendrograms. Also MDS for 2D projection.

In [ ]:
# PARAMETERS — Fig 5 Nucleus Similarity
LINKAGE_METHOD = 'ward'            # 'ward', 'complete', 'average'
DISTANCE_METRIC = 'cosine'         # for pdist (used in MDS / similarity matrix)
MIN_NUCLEUS_SENTENCES_SIMILARITY = 20   # minimum total sentences per nucleus to include

In [ ]:
# ── Tag nuclei on all_sentences_final ─────────────────────────────────────────
df_all5 = df_all.copy()
df_all5 = tag_nuclei(df_all5, text_col='sentence')
df_all5 = tag_techniques(df_all5, text_col='sentence')

# Find nuclei with enough data
nucleus_ids_all = list(NUCLEI.keys())
nucleus_totals_all = {nid: df_all5[f'has_{nid}'].sum() for nid in nucleus_ids_all}
valid_nuc_sim = [n for n in nucleus_ids_all
                 if nucleus_totals_all.get(n, 0) >= MIN_NUCLEUS_SENTENCES_SIMILARITY]
print(f'Nuclei with >= {MIN_NUCLEUS_SENTENCES_SIMILARITY} sentences: {len(valid_nuc_sim)}')
print(valid_nuc_sim)

In [ ]:
# ── Build feature matrices ────────────────────────────────────────────────────

# --- (a) Framing profile (from framing_combined) ---
df_fr5 = df_framing.copy()
df_fr5 = tag_nuclei(df_fr5, text_col='sentence')
frame_types_all = df_fr5['frame_type'].dropna().unique()

framing_matrix = pd.DataFrame(index=valid_nuc_sim, columns=frame_types_all, dtype=float)
for nid in valid_nuc_sim:
    sub = df_fr5[df_fr5[f'has_{nid}']]
    total = max(len(sub), 1)
    for ft in frame_types_all:
        framing_matrix.loc[nid, ft] = (sub['frame_type'] == ft).sum() / total
framing_matrix = framing_matrix.fillna(0).astype(float)

# --- (b) Topic profile (mega_theme from all_sentences_final) ---
mega_themes = df_all5['mega_theme'].dropna().unique()
topic_matrix = pd.DataFrame(index=valid_nuc_sim, columns=mega_themes, dtype=float)
for nid in valid_nuc_sim:
    sub = df_all5[df_all5[f'has_{nid}']]
    total = max(len(sub), 1)
    for theme in mega_themes:
        topic_matrix.loc[nid, theme] = (sub['mega_theme'] == theme).sum() / total
topic_matrix = topic_matrix.fillna(0).astype(float)

# --- (c) Technique profile (from all_sentences_final) ---
tech_cols = [f'has_{t}' for t in TECHNIQUES]
technique_matrix = pd.DataFrame(index=valid_nuc_sim, columns=list(TECHNIQUES.keys()), dtype=float)
for nid in valid_nuc_sim:
    sub = df_all5[df_all5[f'has_{nid}']]
    total = max(len(sub), 1)
    for tid in TECHNIQUES:
        technique_matrix.loc[nid, tid] = sub[f'has_{tid}'].sum() / total
technique_matrix = technique_matrix.fillna(0).astype(float)

print('Feature matrices built:')
print(f'  Framing:   {framing_matrix.shape}')
print(f'  Topic:     {topic_matrix.shape}')
print(f'  Technique: {technique_matrix.shape}')

In [ ]:
# ── Figure 5: Dendrograms + similarity matrix ──────────────────────────────────
nuc_labels = [NUCLEI[n]['label'] for n in valid_nuc_sim]

def plot_dendrogram(ax, matrix, labels, title, method='ward', metric='cosine', color='#0072B2'):
    """Compute linkage and plot a horizontal dendrogram."""
    # Ward linkage requires Euclidean; other methods can use cosine
    if method == 'ward':
        Z = linkage(matrix.values, method='ward')
    else:
        dist = pdist(matrix.values, metric=metric)
        Z = linkage(dist, method=method)
    dendrogram(
        Z,
        orientation='left',
        labels=labels,
        ax=ax,
        link_color_func=lambda k: color,
        leaf_font_size=8,
    )
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel('Distance')
    return Z

fig, axes = plt.subplots(1, 3, figsize=(18, 8))

Z_frame = plot_dendrogram(axes[0], framing_matrix, nuc_labels,
                          'Clustering by framing profile',
                          method=LINKAGE_METHOD, color=WONG_COLORS[4])
Z_topic = plot_dendrogram(axes[1], topic_matrix, nuc_labels,
                          'Clustering by topic (mega-theme) profile',
                          method=LINKAGE_METHOD, color=WONG_COLORS[2])
Z_tech  = plot_dendrogram(axes[2], technique_matrix, nuc_labels,
                          'Clustering by technique profile',
                          method=LINKAGE_METHOD, color=WONG_COLORS[0])

plt.suptitle('Thalamic nucleus similarity (Ward linkage)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
savefig('fig_NB_fig5_dendrograms.png')

In [ ]:
# ── Figure 5 supplement: MDS projection on combined feature matrix ───────────
# Concatenate all three feature vectors for each nucleus
combined_matrix = pd.concat(
    [framing_matrix.reset_index(drop=True),
     topic_matrix.reset_index(drop=True),
     technique_matrix.reset_index(drop=True)],
    axis=1
).values.astype(float)

# Compute pairwise cosine similarity
dist_mat = squareform(pdist(combined_matrix, metric='cosine'))
sim_mat  = 1 - dist_mat  # cosine similarity

# MDS for 2D projection
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, normalized_stress='auto')
coords = mds.fit_transform(dist_mat)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# MDS scatter
for i, (nid, lbl) in enumerate(zip(valid_nuc_sim, nuc_labels)):
    color = NUCLEI[nid]['color']
    ax1.scatter(coords[i, 0], coords[i, 1], c=color, s=120, zorder=3, edgecolors='black', linewidths=0.6)
    ax1.text(coords[i, 0] + 0.005, coords[i, 1] + 0.005, lbl.split('(')[0].strip(),
             fontsize=7, ha='left', va='bottom')
ax1.set_xlabel('MDS dim 1')
ax1.set_ylabel('MDS dim 2')
ax1.set_title('Nucleus similarity — MDS (combined profile)', fontsize=10, fontweight='bold')

# Cosine similarity heatmap
sim_df = pd.DataFrame(sim_mat, index=nuc_labels, columns=nuc_labels)
sns.heatmap(
    sim_df,
    ax=ax2,
    cmap='RdYlGn',
    vmin=0, vmax=1,
    linewidths=0.3,
    cbar_kws={'label': 'Cosine similarity', 'shrink': 0.8},
    xticklabels=[l.split('(')[0].strip() for l in nuc_labels],
    yticklabels=[l.split('(')[0].strip() for l in nuc_labels],
)
ax2.set_title('Nucleus cosine similarity matrix (combined)', fontsize=10, fontweight='bold')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(ax2.get_yticklabels(), fontsize=8)

plt.tight_layout()
savefig('fig_NB_fig5b_mds_similarity.png')

---
## Section 5: Figure 6 — Framing × Topics (Chi-squared residuals)

**Data**: `all_sentences_final.csv` — sentences with both `frame_type` and `topic_clean_v2`.

**Method**:
1. Filter rows with both frame and topic assigned (remove -1 topic, remove null frame)
2. Build contingency table: topics × frame types
3. Compute chi-squared test of independence + standardised Pearson residuals
4. Plot residual heatmap: positive (red) = over-represented, negative (blue) = under-represented

In [ ]:
# PARAMETERS — Fig 6 Framing × Topics
TOP_N_TOPICS_CHI  = 30        # top N topics by count to include in plot
MIN_TOPIC_N       = 20        # minimum sentences per topic
RESIDUAL_CLIP     = 5.0       # clip extreme residuals for colour scale

In [ ]:
# ── Filter and prepare data ───────────────────────────────────────────────────
df_chi = df_all[
    (df_all['bertopic_id_v2'] != -1) &
    df_all['frame_type'].notna() &
    (df_all['frame_type'] != '')
].copy()

print(f'Rows with both topic and frame: {len(df_chi)}')
print('Frame types:', df_chi['frame_type'].value_counts())

# Keep only top-N topics by count
topic_cnts = df_chi['topic_clean_v2'].value_counts()
big_topics_chi = topic_cnts[
    (topic_cnts >= MIN_TOPIC_N)
].head(TOP_N_TOPICS_CHI).index

df_chi_filt = df_chi[df_chi['topic_clean_v2'].isin(big_topics_chi)]
print(f'After filtering: {len(df_chi_filt)} rows, {len(big_topics_chi)} topics')

In [ ]:
# ── Contingency table + chi-squared residuals ─────────────────────────────────
contingency = pd.crosstab(df_chi_filt['topic_clean_v2'], df_chi_filt['frame_type'])

# Chi-squared test
chi2, p_chi, dof, expected = scipy_stats.chi2_contingency(contingency.values)
print(f'Chi-squared: {chi2:.1f}, df={dof}, p={p_chi:.2e}')

# Standardised Pearson residuals: (observed - expected) / sqrt(expected)
residuals = (contingency.values - expected) / np.sqrt(expected)
residuals_df = pd.DataFrame(
    residuals,
    index=contingency.index,
    columns=contingency.columns
)

# Clip for display
residuals_clipped = residuals_df.clip(-RESIDUAL_CLIP, RESIDUAL_CLIP)
print('Residuals computed. Shape:', residuals_df.shape)

In [ ]:
# ── Figure 6: Chi-squared residual heatmap ────────────────────────────────────
fig_h = max(8, len(big_topics_chi) * 0.35)
fig, ax = plt.subplots(figsize=(14, fig_h))

# Shorten topic labels for display
short_labels = [t[:50] for t in residuals_clipped.index]

sns.heatmap(
    residuals_clipped,
    ax=ax,
    cmap='RdBu_r',
    center=0,
    linewidths=0.3,
    linecolor='#cccccc',
    yticklabels=short_labels,
    cbar_kws={
        'label': f'Standardised Pearson residual (clipped ±{RESIDUAL_CLIP})',
        'shrink': 0.7
    },
)
ax.set_xlabel('Frame type')
ax.set_ylabel('BERTopic topic')
ax.set_title(
    f'Framing × Topics: chi-squared residuals\n'
    f'χ²={chi2:.1f}, df={dof}, p={p_chi:.2e}',
    fontsize=11, fontweight='bold'
)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), fontsize=7)

plt.tight_layout()
savefig('fig_NB_fig6_framing_topics_residuals.png')

---
## Section 6: Figure 7 — Techniques Over Time

**Data**: `all_sentences_final.csv` with `tag_techniques()` applied.

**Method**:
1. Tag every sentence with 11 technique booleans
2. For each technique, compute annual proportion (technique sentences / all sentences that year)
3. Mann-Kendall trend per technique
4. Plot: (a) trend lines over time, (b) co-occurrence matrix (how often do pairs appear in same paper)

In [ ]:
# PARAMETERS — Fig 7 Techniques
TECH_MIN_YEAR     = 1980
TECH_MAX_YEAR     = 2026
TECH_SMOOTH_WIN   = 3       # rolling window for trend lines
MIN_TECH_N        = 5       # minimum annual hits to include year in MK test

In [ ]:
# ── Tag techniques on full corpus ─────────────────────────────────────────────
# Re-use df_all5 which already has technique tags from Section 4
# (If running this section standalone, uncomment the line below)
# df_all5 = tag_techniques(df_all.copy(), text_col='sentence')

df_tech = df_all5[
    (df_all5['year'] >= TECH_MIN_YEAR) & (df_all5['year'] <= TECH_MAX_YEAR)
].copy()

# Annual total sentences
ann_total = df_tech.groupby('year').size().rename('total')

# Annual count per technique
tech_annual = {}
for tid in TECHNIQUES:
    counts = df_tech.groupby('year')[f'has_{tid}'].sum()
    prop = counts / ann_total
    tech_annual[tid] = prop.reindex(range(TECH_MIN_YEAR, TECH_MAX_YEAR + 1), fill_value=0)

print('Technique proportions computed.')
for tid, ser in tech_annual.items():
    print(f'  {TECHNIQUES[tid]["label"]:25s}: peak year {ser.idxmax()}, peak proportion {ser.max():.4f}')

In [ ]:
# ── Mann-Kendall trends per technique ─────────────────────────────────────────
mk_tech = {}
years_range = list(range(TECH_MIN_YEAR, TECH_MAX_YEAR + 1))
for tid, ser in tech_annual.items():
    tau, p, sig, label = mann_kendall(years_range, ser.values)
    mk_tech[tid] = {'tau': tau, 'p': p, 'sig': sig, 'trend': label}

print('Technique trends:')
for tid, res in sorted(mk_tech.items(), key=lambda x: x[1]['tau'], reverse=True):
    print(f"  {TECHNIQUES[tid]['label']:25s}: tau={res['tau']:.3f} {res['sig']} ({res['trend']})")

In [ ]:
# ── Figure 7a: Technique trend lines ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 7))

for i, (tid, ser) in enumerate(tech_annual.items()):
    color  = TECHNIQUES[tid]['color']
    label  = TECHNIQUES[tid]['label']
    smooth = ser.rolling(TECH_SMOOTH_WIN, center=True, min_periods=1).mean()
    mk     = mk_tech[tid]
    lbl    = f"{label} (τ={mk['tau']:.2f}{mk['sig']})"
    ax.plot(years_range, smooth, color=color, linewidth=1.8, label=lbl)

ax.set_xlabel('Year')
ax.set_ylabel('Proportion of all thalamus sentences')
ax.set_title('Recording technique prevalence over time (3-year rolling mean)', fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='upper left', framealpha=0.5, ncol=2)
ax.set_xlim(TECH_MIN_YEAR, TECH_MAX_YEAR)

plt.tight_layout()
savefig('fig_NB_fig7a_techniques_over_time.png')

In [ ]:
# ── Figure 7b: Technique co-occurrence matrix (paper-level) ──────────────────
# Group by pmid: if a paper has any sentence tagged with a technique, mark it as 1
tech_cols = [f'has_{tid}' for tid in TECHNIQUES]
paper_tech = df_all5.groupby('pmid')[tech_cols].max()  # 1 if any sentence in paper matches

# Co-occurrence: count papers where both techniques appear
n_papers = len(paper_tech)
co_mat = np.zeros((len(TECHNIQUES), len(TECHNIQUES)), dtype=float)
tech_list = list(TECHNIQUES.keys())
for i, ta in enumerate(tech_list):
    for j, tb in enumerate(tech_list):
        both = (paper_tech[f'has_{ta}'] & paper_tech[f'has_{tb}']).sum()
        co_mat[i, j] = both / n_papers

co_df = pd.DataFrame(
    co_mat,
    index=[TECHNIQUES[t]['label'] for t in tech_list],
    columns=[TECHNIQUES[t]['label'] for t in tech_list]
)

fig, ax = plt.subplots(figsize=(11, 9))
# Mask diagonal for cleaner display
mask = np.eye(len(tech_list), dtype=bool)
sns.heatmap(
    co_df,
    ax=ax,
    mask=mask,
    cmap='YlOrRd',
    linewidths=0.3,
    annot=True,
    fmt='.3f',
    annot_kws={'size': 7},
    cbar_kws={'label': 'Proportion of papers', 'shrink': 0.8}
)
ax.set_title('Technique co-occurrence in papers (proportion of all papers)', fontsize=11, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), fontsize=8)

plt.tight_layout()
savefig('fig_NB_fig7b_technique_cooccurrence.png')

---
## Section 7: Figure 8 — Task Extraction

**Data**: `all_sentences_final.csv`.

**Method**:
1. Apply 10 regex patterns for experimental paradigm categories
2. Compute annual prevalence of each task type
3. Build task × nucleus cross-tabulation

In [ ]:
# PARAMETERS — Fig 8 Tasks
TASK_PATTERNS = {
    'Working memory':      r'working memory|n-back|delayed match|visuospatial memory',
    'Spatial memory':      r'spatial memory|spatial navigation|Morris water maze|radial arm|place cell',
    'Fear conditioning':   r'fear conditioning|fear extinction|cued fear|contextual fear|CFC\b',
    'Sensory discrimination': r'sensory discrimination|detection threshold|psychophysics|perceptual',
    'Attention':           r'attention|attentional|selective attention|sustained attention|attentional blink',
    'Motor tasks':         r'motor learning|motor task|reaching task|rotarod|gait analysis',
    'Pain assessment':     r'von Frey|hot plate|tail flick|mechanical allodynia|thermal hyperalgesia',
    'Sleep recording':     r'sleep recording|polysomnograph|sleep stage|NREM|REM sleep|sleep EEG',
    'Reward/motivation':   r'reward|reinforcement|sucrose preference|conditioned place preference|dopamine reward',
    'Decision making':     r'decision making|two-alternative|forced choice|Iowa gambling|reversal learning',
}
TASK_MIN_YEAR = 1980
TASK_MAX_YEAR = 2026

In [ ]:
# ── Tag tasks on full corpus ───────────────────────────────────────────────────
df_tasks = df_all.copy()

for task_name, pattern in TASK_PATTERNS.items():
    col = 'task_' + task_name.lower().replace(' ', '_').replace('/', '_')
    df_tasks[col] = df_tasks['sentence'].str.contains(
        pattern, case=False, regex=True, na=False
    )

task_cols = ['task_' + t.lower().replace(' ', '_').replace('/', '_') for t in TASK_PATTERNS]

print('Task match counts:')
for col, name in zip(task_cols, TASK_PATTERNS):
    print(f'  {name:30s}: {df_tasks[col].sum()}')

In [ ]:
# ── Annual task proportions ───────────────────────────────────────────────────
df_tasks_yr = df_tasks[
    (df_tasks['year'] >= TASK_MIN_YEAR) & (df_tasks['year'] <= TASK_MAX_YEAR)
]
ann_total_tasks = df_tasks_yr.groupby('year').size().rename('total')

task_proportions = {}
for col, name in zip(task_cols, TASK_PATTERNS):
    counts = df_tasks_yr.groupby('year')[col].sum()
    prop   = (counts / ann_total_tasks).reindex(
        range(TASK_MIN_YEAR, TASK_MAX_YEAR + 1), fill_value=0
    )
    task_proportions[name] = prop

print('Annual task proportions computed.')

In [ ]:
# ── Task × Nucleus cross-tabulation ───────────────────────────────────────────
# Tag nuclei on task-annotated dataframe
df_tasks_nuc = tag_nuclei(df_tasks.copy(), text_col='sentence')

# Build matrix: rows = nuclei, cols = task types
task_nuc_matrix = pd.DataFrame(
    index=valid_nuc_sim,  # nuclei defined in Section 4
    columns=list(TASK_PATTERNS.keys()),
    dtype=float
)
for nid in valid_nuc_sim:
    sub   = df_tasks_nuc[df_tasks_nuc[f'has_{nid}']]
    total = max(len(sub), 1)
    for col, name in zip(task_cols, TASK_PATTERNS):
        task_nuc_matrix.loc[nid, name] = sub[col].sum() / total

task_nuc_matrix = task_nuc_matrix.fillna(0).astype(float)
print('Task × Nucleus matrix:')
print(task_nuc_matrix.round(3))

In [ ]:
# ── Figure 8: (a) Task trends over time + (b) Task × Nucleus heatmap ─────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Panel A: trend lines
years_tasks = list(range(TASK_MIN_YEAR, TASK_MAX_YEAR + 1))
for i, (name, ser) in enumerate(task_proportions.items()):
    color  = WONG_COLORS[i % len(WONG_COLORS)]
    smooth = ser.rolling(3, center=True, min_periods=1).mean()
    tau, p, sig, trend = mann_kendall(years_tasks, smooth.values)
    lbl = f"{name} (τ={tau:.2f}{sig})"
    ax1.plot(years_tasks, smooth, color=color, linewidth=1.8, label=lbl)

ax1.set_xlabel('Year')
ax1.set_ylabel('Proportion of all thalamus sentences')
ax1.set_title('Experimental task prevalence over time', fontsize=11, fontweight='bold')
ax1.legend(fontsize=7, loc='upper left', framealpha=0.5)
ax1.set_xlim(TASK_MIN_YEAR, TASK_MAX_YEAR)

# Panel B: Task × Nucleus heatmap
nuc_labels_sim = [NUCLEI[n]['label'] for n in valid_nuc_sim]
sns.heatmap(
    task_nuc_matrix,
    ax=ax2,
    cmap='YlOrBr',
    linewidths=0.4,
    annot=True,
    fmt='.3f',
    annot_kws={'size': 7},
    yticklabels=nuc_labels_sim,
    cbar_kws={'label': 'Proportion of nucleus sentences', 'shrink': 0.8}
)
ax2.set_xlabel('Task type')
ax2.set_ylabel('Thalamic nucleus')
ax2.set_title('Task × Nucleus (proportion)', fontsize=11, fontweight='bold')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(ax2.get_yticklabels(), fontsize=8)

plt.tight_layout()
savefig('fig_NB_fig8_tasks.png')

---
## Section 8: Figure 9 — Confound Analysis (Logistic Regression)

**Data**: `framing_combined.csv` — sentences with `frame_type` in {hub_node, relay}.

**Method**:
1. Filter to hub vs. relay sentences only
2. Tag techniques (fMRI, electrophysiology, lesion studies)
3. Fit logistic regression: `is_hub ~ year_c + fMRI + electrophysiology + lesion`
4. Extract odds ratios + 95% CIs from the fitted model
5. Plot forest plot of ORs

**Key result** (from prior analysis): year OR ≈ 1.17/year (p < 1e-30), fMRI OR ≈ 3.71 (p ≈ 0.009).

In [ ]:
# PARAMETERS — Fig 9 Confound
OUTCOME_FRAME_1     = 'hub_node'        # coded as 1 (positive outcome)
OUTCOME_FRAME_2     = 'relay'           # coded as 0
YEAR_CENTER         = 2000              # subtract from year for interpretability
TECHNIQUE_PREDICTORS = ['fMRI', 'electrophysiology', 'lesion']  # keys in TECHNIQUES dict

In [ ]:
# ── Filter hub vs relay ───────────────────────────────────────────────────────
df_log = df_framing[
    df_framing['frame_type'].isin([OUTCOME_FRAME_1, OUTCOME_FRAME_2])
].copy()

print(f'Hub sentences: {(df_log["frame_type"] == OUTCOME_FRAME_1).sum()}')
print(f'Relay sentences: {(df_log["frame_type"] == OUTCOME_FRAME_2).sum()}')

# Outcome: 1 = hub, 0 = relay
df_log['is_hub'] = (df_log['frame_type'] == OUTCOME_FRAME_1).astype(int)

# Center year
df_log['year_c'] = df_log['year'] - YEAR_CENTER

# Tag techniques
df_log = tag_techniques(df_log, text_col='sentence')

# Convert boolean technique columns to int
for tid in TECHNIQUE_PREDICTORS:
    df_log[f'has_{tid}'] = df_log[f'has_{tid}'].astype(int)

df_log = df_log.dropna(subset=['year_c', 'is_hub'])
print(f'Final model dataset: {len(df_log)} rows')

In [ ]:
# ── Fit logistic regression with statsmodels ──────────────────────────────────
try:
    import statsmodels.api as sm
    STATSMODELS_AVAILABLE = True
except ImportError:
    print('statsmodels not available; falling back to sklearn LogisticRegression')
    STATSMODELS_AVAILABLE = False

predictor_cols = ['year_c'] + [f'has_{tid}' for tid in TECHNIQUE_PREDICTORS]
X = df_log[predictor_cols].copy()
y = df_log['is_hub']

if STATSMODELS_AVAILABLE:
    X_const = sm.add_constant(X)
    model = sm.GLM(y, X_const, family=sm.families.Binomial())
    result = model.fit()
    print(result.summary())

    # Extract coefficients, CIs, p-values
    coefs = result.params.drop('const')
    conf  = result.conf_int().drop('const')
    pvals = result.pvalues.drop('const')
    
    # Odds ratios
    ors     = np.exp(coefs)
    ci_low  = np.exp(conf[0])
    ci_high = np.exp(conf[1])
    
    pred_labels = [
        f'Year (centered {YEAR_CENTER})',
    ] + [TECHNIQUES[tid]['label'] for tid in TECHNIQUE_PREDICTORS]

else:
    # Fallback: scipy logistic regression — manual CI via bootstrap approximation
    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(max_iter=1000, solver='lbfgs')
    clf.fit(X, y)
    coefs = pd.Series(clf.coef_[0], index=predictor_cols)
    ors = np.exp(coefs)
    se_approx = np.sqrt(np.diag(np.linalg.inv(X.T @ X))) * 1.5  # rough SE
    ci_low  = np.exp(coefs - 1.96 * se_approx)
    ci_high = np.exp(coefs + 1.96 * se_approx)
    pvals = pd.Series([np.nan] * len(coefs), index=predictor_cols)
    pred_labels = [
        f'Year (centered {YEAR_CENTER})',
    ] + [TECHNIQUES[tid]['label'] for tid in TECHNIQUE_PREDICTORS]
    print('Fallback ORs (approximate CIs):')
    print(pd.DataFrame({'OR': ors, 'CI_low': ci_low, 'CI_high': ci_high}))

In [ ]:
# ── Figure 9: Forest plot of odds ratios ──────────────────────────────────────
n_pred = len(predictor_cols)
y_pos  = list(range(n_pred))

fig, ax = plt.subplots(figsize=(9, 4))

for i, (col, lbl) in enumerate(zip(predictor_cols, pred_labels)):
    or_val = float(ors[col])
    lo     = float(ci_low[col]) if hasattr(ci_low, '__getitem__') else float(ci_low.iloc[i])
    hi     = float(ci_high[col]) if hasattr(ci_high, '__getitem__') else float(ci_high.iloc[i])
    p      = float(pvals[col]) if col in pvals.index else np.nan
    
    color = '#D55E00' if or_val > 1 else '#0072B2'
    
    # Error bar: CI
    ax.plot([lo, hi], [i, i], color=color, linewidth=2, zorder=2)
    ax.scatter([or_val], [i], color=color, s=80, zorder=3, edgecolors='black', linewidths=0.8)
    
    # P-value annotation
    if not np.isnan(p):
        sig_str = ('***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns')
        ax.text(hi + 0.05, i, f'OR={or_val:.2f} [{lo:.2f}–{hi:.2f}] {sig_str}',
                va='center', fontsize=8)

# Reference line at OR = 1
ax.axvline(1.0, color='black', linewidth=1, linestyle='--', alpha=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(pred_labels, fontsize=9)
ax.invert_yaxis()
ax.set_xscale('log')
ax.set_xlabel('Odds Ratio (log scale)')
ax.set_title(
    f'Logistic regression: {OUTCOME_FRAME_1} vs {OUTCOME_FRAME_2}\n'
    f'(outcome = 1 if {OUTCOME_FRAME_1})',
    fontsize=11, fontweight='bold'
)

plt.tight_layout()
savefig('fig_NB_fig9_confound_logistic.png')

---
## Appendix: Conclusion Themes Analysis

**Data**: `conclusion_themes_v2.csv` — conclusion sentences with 10 binary theme columns.

This section provides a supplementary look at how framing language maps onto conclusion section themes.

In [ ]:
# ── Load conclusion themes ────────────────────────────────────────────────────
df_conc = pd.read_csv(PROC / 'conclusion_themes_v2.csv', low_memory=False)
print(f'Loaded conclusion_themes_v2.csv: {df_conc.shape}')
print('Columns:', list(df_conc.columns))

# Identify the 10 binary theme columns
# (all boolean columns that are not standard metadata)
META_COLS = ['pmid', 'year', 'sentence', 'section', 'frame_type']
theme_cols = [c for c in df_conc.columns if c not in META_COLS]
print(f'Theme columns ({len(theme_cols)}): {theme_cols}')

In [ ]:
# ── Annual prevalence of each conclusion theme ─────────────────────────────────
# Convert theme columns to bool if stored as object ('True'/'False' strings)
for col in theme_cols:
    if df_conc[col].dtype == object:
        df_conc[col] = df_conc[col].map({'True': True, 'False': False, True: True, False: False})

# Annual proportion per theme
df_conc_yr = df_conc.dropna(subset=['year'])
ann_conc_total = df_conc_yr.groupby('year').size()

fig, ax = plt.subplots(figsize=(13, 6))

for i, col in enumerate(theme_cols):
    color  = WONG_COLORS[i % len(WONG_COLORS)]
    counts = df_conc_yr.groupby('year')[col].sum()
    prop   = counts / ann_conc_total
    smooth = prop.rolling(3, center=True, min_periods=1).mean()
    tau, p, sig, trend = mann_kendall(
        smooth.index.tolist(), smooth.values
    )
    lbl = f"{col} (τ={tau:.2f}{sig})"
    ax.plot(smooth.index, smooth.values, color=color, linewidth=1.8, label=lbl)

ax.set_xlabel('Year')
ax.set_ylabel('Proportion of conclusion sentences')
ax.set_title('Conclusion theme prevalence over time', fontsize=11, fontweight='bold')
ax.legend(fontsize=7, loc='upper left', framealpha=0.5, ncol=2)

plt.tight_layout()
savefig('fig_NB_app_conclusion_themes.png')

In [ ]:
# ── Summary: all saved figures ────────────────────────────────────────────────
print('Figures saved to:', PUB)
saved = sorted(PUB.glob('fig_NB_*.png'))
for f in saved:
    print(f'  {f.name}')